In [ ]:
# UPLOAD 5 FILE CSV TỪ MÁY TÍNH LÊN COLAB
from google.colab import files
import pandas as pd
import numpy as np

print("Hãy bấm nút 'Choose Files' bên dưới để chọn cả 5 file CSV từ máy tính:")
uploaded = files.upload()

Hãy bấm nút 'Choose Files' bên dưới để chọn cả 5 file CSV từ máy tính:


Saving events.csv to events.csv
Saving products.csv to products.csv
Saving transactions.csv to transactions.csv
Saving campaigns.csv to campaigns.csv
Saving customers.csv to customers.csv


In [ ]:
# ĐỌC DỮ LIỆU VÀO PANDAS DATAFRAME
campaigns = pd.read_csv('campaigns.csv')
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')
transactions = pd.read_csv('transactions.csv')
events = pd.read_csv('events.csv')

# KIỂM TRA KÍCH THƯỚC DỮ LIỆU (SHAPE)
print("\nTải dữ liệu hoàn tất! Kích thước từng bảng (Dòng, Cột):")
print('customers   :', customers.shape)
print('events      :', events.shape)
print('transactions:', transactions.shape)
print('products    :', products.shape)
print('campaigns   :', campaigns.shape)


Tải dữ liệu hoàn tất! Kích thước từng bảng (Dòng, Cột):
customers   : (100000, 7)
events      : (2000000, 12)
transactions: (103127, 9)
products    : (2000, 6)
campaigns   : (50, 7)


In [ ]:
# Hàm kiểm tra nhanh cấu trúc data
def deep_inspect(df, name):
    print(f"=== {name.upper()} DATASET ===")
    print(f"Shape: {df.shape}\n")

    summary = pd.DataFrame({
        'dtypes': df.dtypes,
        'null_count': df.isnull().sum(),
        'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
        'nunique': df.nunique(),
        'sample': df.iloc[0] if len(df) > 0 else None
    })

    print(summary)
    print(f"\nDuplicates: {df.duplicated().sum()}")
    print("-" * 50 + "\n")

# List data
dataset_list = [
    (customers, 'customers'),
    (events, 'events'),
    (transactions, 'transactions'),
    (products, 'products'),
    (campaigns, 'campaigns')
]

for df, name in dataset_list:
    deep_inspect(df, name)



=== CUSTOMERS DATASET ===
Shape: (100000, 7)

                     dtypes  null_count  null_pct  nunique      sample
customer_id           int64           0       0.0   100000           1
signup_date          object           0       0.0     1095  2021-04-08
country              object           0       0.0        7          BR
age                   int64           0       0.0       53          48
gender               object           0       0.0        3        Male
loyalty_tier         object           0       0.0        4      Bronze
acquisition_channel  object           0       0.0        5    Referral

Duplicates: 0
--------------------------------------------------

=== EVENTS DATASET ===
Shape: (2000000, 12)

                       dtypes  null_count  null_pct  nunique  \
event_id                int64           0      0.00  2000000   
timestamp              object           0      0.00  1977065   
customer_id             int64           0      0.00   100000   
session_id        

In [ ]:
# Kiểm tra chỉ số thống kê của các biến định lượng
print("=== TRANSACTIONS DESCRIBE ===")
print(transactions[['gross_revenue', 'quantity', 'discount_applied', 'refund_flag']].describe().T[['min', 'mean', 'max', '50%']])

print("\n=== EVENTS DESCRIBE ===")
print(events[['session_duration_sec']].describe().T[['min', 'mean', 'max', '50%']])

print("\n=== PRODUCTS DESCRIBE ===")
print(products[['base_price']].describe().T[['min', 'mean', 'max', '50%']])

print("\n=== CUSTOMERS DESCRIBE ===")
print(customers[['age']].describe().T[['min', 'mean', 'max', '50%']])

=== TRANSACTIONS DESCRIBE ===
                     min       mean      max   50%
gross_revenue    -783.04  90.355493  1858.32  68.0
quantity            1.00   1.381539     4.00   1.0
discount_applied    0.00   0.041379     0.20   0.0
refund_flag         0.00   0.029372     1.00   0.0

=== EVENTS DESCRIBE ===
                      min        mean     max    50%
session_duration_sec  0.1  130.642617  7533.8  101.7

=== PRODUCTS DESCRIBE ===
             min      mean     max     50%
base_price  5.11  72.16665  464.58  61.245

=== CUSTOMERS DESCRIBE ===
      min      mean   max   50%
age  18.0  35.22437  70.0  35.0


In [ ]:
# 1. Gộp bảng giao dịch sạch với khách hàng, sản phẩm, chiến dịch
df_trans_clean = transactions.dropna(subset=['gross_revenue', 'product_id']).copy()
df_trans_clean['timestamp'] = pd.to_datetime(df_trans_clean['timestamp'])
df_trans_clean['product_id'] = df_trans_clean['product_id'].astype(int)

# Giữ lại đơn refund (net_revenue = 0)
df_trans_clean['net_revenue'] = np.where(
    df_trans_clean['refund_flag'] == 1,
    0,
    df_trans_clean['gross_revenue']
)

# 2. Chuẩn hóa events & ngày tháng
events_clean = events.copy()
events_clean['product_id'] = events_clean['product_id'].fillna(-1).astype(int)
events_clean['device_type'] = events_clean['device_type'].fillna('Unknown')
events_clean['timestamp'] = pd.to_datetime(events_clean['timestamp'])

customers['signup_date'] = pd.to_datetime(customers['signup_date'])
products['launch_date'] = pd.to_datetime(products['launch_date'])
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['end_date'] = pd.to_datetime(campaigns['end_date'])

# 3. Gộp Master Data
master_df = df_trans_clean.merge(customers, on='customer_id', how='left') \
                          .merge(products, on='product_id', how='left') \
                          .merge(campaigns, on='campaign_id', how='left')

# 4. Xuất file kết quả
master_df.to_csv('processed_master_data.csv', index=False)
events_clean.to_csv('processed_events_data.csv', index=False)

print("Đã xuất file processed_master_data.csv và processed_events_data.csv!")

Đã xuất file processed_master_data.csv và processed_events_data.csv!


In [ ]:
# 1. Đọc dữ liệu master đã làm sạch
master_df = pd.read_csv('processed_master_data.csv')
events_clean = pd.read_csv('processed_events_data.csv')
customers = pd.read_csv('customers.csv')

# Ép kiểu thời gian
master_df['timestamp'] = pd.to_datetime(master_df['timestamp'])
events_clean['timestamp'] = pd.to_datetime(events_clean['timestamp'])

# Mốc thời gian chốt dữ liệu phân tích (Snapshot Date)
snapshot_date = master_df['timestamp'].max() + pd.Timedelta(days=1)

# 2. Tính toán bộ chỉ số RFM & Đặc trưng giao dịch (từ master_df)
rfm = master_df.groupby('customer_id').agg(
    Recency=('timestamp', lambda x: (snapshot_date - x.max()).days),
    Frequency=('transaction_id', 'nunique'),
    Net_Monetary=('net_revenue', 'sum'),
    Avg_Order_Value=('net_revenue', 'mean'),
    Total_Refunds=('refund_flag', 'sum'),
    Favorite_Category=('category', lambda x: x.mode()[0] if not x.empty else 'Unknown')
).reset_index()

# 3. Tính toán bộ chỉ số Tương tác & Hành vi (từ events_clean)
event_features = events_clean.groupby('customer_id').agg(
    Total_Sessions=('session_id', 'nunique'),
    Avg_Session_Duration=('session_duration_sec', 'mean'),
    Total_Views=('event_type', lambda x: (x == 'view').sum()),
    Total_Add_To_Cart=('event_type', lambda x: (x == 'add_to_cart').sum()),
    Total_Bounces=('event_type', lambda x: (x == 'bounce').sum()),
    Mobile_Ratio=('device_type', lambda x: (x == 'mobile').mean())
).reset_index()

# Tỷ lệ chuyển đổi giỏ hàng (Cart Conversion Rate)
event_features['Cart_Conversion_Rate'] = np.where(
    event_features['Total_Add_To_Cart'] > 0,
    rfm.set_index('customer_id')['Frequency'].reindex(event_features['customer_id']).fillna(0).values / event_features['Total_Add_To_Cart'],
    0
)

# 4. Ghép toàn bộ vào danh sách 100,000 khách hàng (Master Feature Store)
customer_features = customers.merge(rfm, on='customer_id', how='left') \
                             .merge(event_features, on='customer_id', how='left')

# Điền giá trị khuyết cho khách hàng chưa từng phát sinh đơn hàng
customer_features['Frequency'] = customer_features['Frequency'].fillna(0)
customer_features['Net_Monetary'] = customer_features['Net_Monetary'].fillna(0)
customer_features['Total_Sessions'] = customer_features['Total_Sessions'].fillna(0)
customer_features['Has_Purchased'] = np.where(customer_features['Frequency'] > 0, 1, 0)

# 5. Lưu bảng dữ liệu đặc trưng khách hàng ra file
customer_features.to_csv('customer_features_master.csv', index=False)

print(f"Tổng số khách hàng: {customer_features.shape[0]:,}")
print(f"Số lượng biến đặc trưng (cột): {customer_features.shape[1]}")
print("\nXem trước 5 dòng đầu:")
print(customer_features.head())

Tổng số khách hàng: 100,000
Số lượng biến đặc trưng (cột): 21

Xem trước 5 dòng đầu:
   customer_id signup_date country  age  gender loyalty_tier  \
0            1  2021-04-08      BR   48    Male       Bronze   
1            2  2023-04-28      IN   36  Female       Silver   
2            3  2022-12-18      UK   35  Female       Silver   
3            4  2022-04-26      US   45    Male       Silver   
4            5  2022-04-20      IN   53    Male       Silver   

  acquisition_channel  Recency  Frequency  Net_Monetary  ...  Total_Refunds  \
0            Referral      NaN        0.0          0.00  ...            NaN   
1             Organic   1066.0        1.0         60.09  ...            0.0   
2             Organic      NaN        0.0          0.00  ...            NaN   
3         Paid Search      NaN        0.0          0.00  ...            NaN   
4             Organic      NaN        0.0          0.00  ...            NaN   

   Favorite_Category Total_Sessions  Avg_Session_Durati

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Lọc tập khách hàng đã từng phát sinh giao dịch
df_purchased = customer_features[customer_features['Has_Purchased'] == 1].copy()

# Chọn bộ biến phục vụ gom cụm RFM
rfm_cols = ['Recency', 'Frequency', 'Net_Monetary']

# 2. Xử lý độ lệch (Log Transformation) để chuẩn hóa dữ liệu
df_log = np.log1p(df_purchased[rfm_cols])

# 3. Chuẩn hóa thang đo (Standardization)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_log)

# 4. Huấn luyện mô hình K-Means với K = 4 cụm
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_purchased['Cluster'] = kmeans.fit_predict(df_scaled)

# 5. Đặt tên phân đoạn kinh doanh (Business Segment Mapping)
# Tính trung bình RFM của từng cụm để gán nhãn chính xác
cluster_summary = df_purchased.groupby('Cluster')[rfm_cols].mean()
print("=== THỐNG KÊ TRUNG BÌNH CÁC CỤM K-MEANS ===")
print(cluster_summary)

# Áp dụng hàm phân loại tự động dựa trên rank của chỉ số trong các cụm
# VIP: Recency thấp, Frequency cao, Monetary cao
# At-Risk: Recency cao, Frequency cao, Monetary cao
# New/Promising: Recency thấp, Frequency thấp
# Hibernating/Low-value: Recency cao, Frequency thấp, Monetary thấp

segment_map = {}
for cluster_id in cluster_summary.index:
    r = cluster_summary.loc[cluster_id, 'Recency']
    f = cluster_summary.loc[cluster_id, 'Frequency']
    m = cluster_summary.loc[cluster_id, 'Net_Monetary']

    if f > cluster_summary['Frequency'].median() and m > cluster_summary['Net_Monetary'].median():
        if r < cluster_summary['Recency'].median():
            segment_map[cluster_id] = 'Champions / VIP'
        else:
            segment_map[cluster_id] = 'At-Risk High Value'
    else:
        if r < cluster_summary['Recency'].median():
            segment_map[cluster_id] = 'New & Promising'
        else:
            segment_map[cluster_id] = 'Hibernating / Low Value'

df_purchased['Segment'] = df_purchased['Cluster'].map(segment_map)

# 6. Cập nhật phân đoạn về bảng Master 100,000 khách hàng
customer_features['Segment'] = 'Non-Purchaser' # Nhóm chưa mua hàng
customer_features.loc[df_purchased.index, 'Segment'] = df_purchased['Segment']

print("\n=== SỐ LƯỢNG KHÁCH HÀNG THEO TỪNG PHÂN ĐOẠN ===")
print(customer_features['Segment'].value_counts())

=== THỐNG KÊ TRUNG BÌNH CÁC CỤM K-MEANS ===
            Recency  Frequency  Net_Monetary
Cluster                                     
0        602.251986   1.000000    118.607453
1        552.642711   1.036202     18.251364
2        391.185980   2.401248    230.203099
3         41.589108   1.785433    163.947990

=== SỐ LƯỢNG KHÁCH HÀNG THEO TỪNG PHÂN ĐOẠN ===
Segment
Non-Purchaser              39909
Hibernating / Low Value    33727
Champions / VIP            26364
Name: count, dtype: int64


In [ ]:
!pip install lifetimes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 9.6 MB/s eta 0:00:00


In [ ]:
from lifetimes import BetaGeoFitter, GammaGammaFitter

# 1. Chuẩn bị dữ liệu RFM dạng Summary cho lifetimes
# Lifetimes yêu cầu: frequency (số lần mua lặp lại = F - 1), recency (số ngày giữa đơn đầu và đơn cuối), T (tuổi khách hàng = snapshot - đơn đầu)
tx_clean = master_df[master_df['net_revenue'] > 0].copy()

# Mốc snapshot
max_date = tx_clean['timestamp'].max()

# Tính các biến cho Lifetimes
clv_summary = tx_clean.groupby('customer_id').agg(
    first_order=('timestamp', 'min'),
    last_order=('timestamp', 'max'),
    frequency=('transaction_id', 'nunique'),
    monetary_value=('net_revenue', 'mean')
).reset_index()

# Chuyển đổi sang chuẩn lifetimes
clv_summary['recency'] = (clv_summary['last_order'] - clv_summary['first_order']).dt.days
clv_summary['T'] = (max_date - clv_summary['first_order']).dt.days
clv_summary['frequency'] = clv_summary['frequency'] - 1 # Chỉ tính số lần repeat purchase

# Lọc các khách hàng có giao dịch hợp lệ
clv_data = clv_summary[clv_summary['T'] > 0].copy()

# 2. Huấn luyện BG/NBD Model
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(clv_data['frequency'], clv_data['recency'], clv_data['T'])

# Dự báo số lượng đơn hàng trong 365 ngày tới (12 tháng)
clv_data['predicted_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    365, clv_data['frequency'], clv_data['recency'], clv_data['T']
)

# 3. Huấn luyện Gamma-Gamma Model (chỉ trên nhóm khách hàng có repeat purchase > 0)
returning_customers = clv_data[(clv_data['frequency'] > 0) & (clv_data['monetary_value'] > 0)]

ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])

# 4. Tính toán CLV dự báo 12 tháng
clv_data['predicted_clv_12m'] = ggf.customer_lifetime_value(
    bgf,
    clv_data['frequency'],
    clv_data['recency'],
    clv_data['T'],
    clv_data['monetary_value'],
    time=12, # 12 tháng
    discount_rate=0.01
)

# Ghép kết quả CLV dự báo vào bảng customer_features master
customer_features = customer_features.merge(
    clv_data[['customer_id', 'predicted_purchases_12m', 'predicted_clv_12m']],
    on='customer_id',
    how='left'
)

customer_features['predicted_purchases_12m'] = customer_features['predicted_purchases_12m'].fillna(0)
customer_features['predicted_clv_12m'] = customer_features['predicted_clv_12m'].fillna(0)

print("=== HOÀN THÀNH DỰ BÁO CLV 12 THÁNG ===")
print(customer_features.groupby('Segment')[['Net_Monetary', 'predicted_clv_12m']].mean())

=== HOÀN THÀNH DỰ BÁO CLV 12 THÁNG ===
                         Net_Monetary  predicted_clv_12m
Segment                                                 
Champions / VIP            211.053352          27.137911
Hibernating / Low Value     90.908137          -7.929606
Non-Purchaser                0.000000           0.000000


In [ ]:
from lifetimes import BetaGeoFitter, GammaGammaFitter

# 1. Chuẩn bị dữ liệu chỉ gồm đơn hàng phát sinh doanh thu dương
tx_clean = master_df[master_df['net_revenue'] > 0].copy()
max_date = tx_clean['timestamp'].max()

clv_summary = tx_clean.groupby('customer_id').agg(
    first_order=('timestamp', 'min'),
    last_order=('timestamp', 'max'),
    frequency=('transaction_id', 'nunique'),
    monetary_value=('net_revenue', 'mean')
).reset_index()

clv_summary['recency'] = (clv_summary['last_order'] - clv_summary['first_order']).dt.days
clv_summary['T'] = (max_date - clv_summary['first_order']).dt.days
clv_summary['frequency'] = clv_summary['frequency'] - 1 # Lifetimes dùng repeat purchase

clv_data = clv_summary[clv_summary['T'] > 0].copy()

# 2. Fit BG/NBD Model (Dự báo số lượng giao dịch)
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(clv_data['frequency'], clv_data['recency'], clv_data['T'])

clv_data['predicted_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    365, clv_data['frequency'], clv_data['recency'], clv_data['T']
)

# 3. Fit Gamma-Gamma Model (Chỉ fit trên nhóm có repeat purchase > 0)
returning_customers = clv_data[(clv_data['frequency'] > 0) & (clv_data['monetary_value'] > 0)]

ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])

# Dự báo giá trị đơn hàng trung bình kỳ vọng (Expected Average Value)
clv_data['expected_avg_order_value'] = ggf.conditional_expected_average_profit(
    clv_data['frequency'], clv_data['monetary_value']
)

# 4. Tính CLV dự báo = (Số đơn dự báo) * (Giá trị đơn dự báo)
clv_data['predicted_clv_12m'] = clv_data['predicted_purchases_12m'] * clv_data['expected_avg_order_value']

# Ép các giá trị âm hoặc vô lý về 0
clv_data['predicted_clv_12m'] = np.where(clv_data['predicted_clv_12m'] < 0, 0, clv_data['predicted_clv_12m'])

# 5. Cập nhật vào bảng Master Features
if 'predicted_clv_12m' in customer_features.columns:
    customer_features = customer_features.drop(columns=['predicted_purchases_12m', 'predicted_clv_12m'])

customer_features = customer_features.merge(
    clv_data[['customer_id', 'predicted_purchases_12m', 'predicted_clv_12m']],
    on='customer_id',
    how='left'
)

customer_features['predicted_purchases_12m'] = customer_features['predicted_purchases_12m'].fillna(0)
customer_features['predicted_clv_12m'] = customer_features['predicted_clv_12m'].fillna(0)

# Lưu lại bản master đầy đủ các chỉ số
customer_features.to_csv('final_customer_analytics_master.csv', index=False)

print("=== HOÀN THÀNH DỰ BÁO CLV 12 THÁNG (ĐÃ SỬA LỖI ÂM) ===")
print(customer_features.groupby('Segment')[['Net_Monetary', 'predicted_purchases_12m', 'predicted_clv_12m']].mean())

=== HOÀN THÀNH DỰ BÁO CLV 12 THÁNG (ĐÃ SỬA LỖI ÂM) ===
                         Net_Monetary  predicted_purchases_12m  \
Segment                                                          
Champions / VIP            211.053352                 0.336060   
Hibernating / Low Value     90.908137                 0.211089   
Non-Purchaser                0.000000                 0.000000   

                         predicted_clv_12m  
Segment                                     
Champions / VIP                  31.148181  
Hibernating / Low Value           0.018264  
Non-Purchaser                     0.000000  


In [ ]:
from google.colab import files

# Tự động tải file master hoàn chỉnh về máy tính
files.download('final_customer_analytics_master.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>